# Libraries

In [29]:
from datetime import datetime as dt
from langchain.tools import tool
from langchain_community.document_loaders import JSONLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM, ChatOllama
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field
from typing import TypedDict, Optional, List, Annotated
import json
import matplotlib.pyplot as plt
import numpy as np
import operator
import os
import pandas as pd
import seaborn as sns



# Préparation des données

In [30]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from langchain_core.documents import Document

def charger_et_preparer_csv(file_path="PAL_final.csv"):
    print(f"Chargement des données depuis {file_path}...")
    
    # Lecture du CSV avec pandas (gestion des séparateurs et de l'encodage)
    df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig')
    
    # Remplacer les valeurs manquantes (NaN) par None ou des chaînes vides pour éviter les bugs
    df = df.replace({np.nan: None})
    
    documents = []
    
    for index, row in df.iterrows():
        # --- 1. LE CONTENU SÉMANTIQUE (Ce que l'IA va "lire" et chercher par similarité) ---
        titre = row.get("Titre VF", "")
        auteur = row.get("Auteur(s)", "")
        resume = row.get("Résumé", "Aucun résumé disponible.")
        
        page_content = f"Titre: {titre}\nAuteur: {auteur}\nRésumé: {resume}"
        
        # --- 2. LES MÉTADONNÉES (Ce qui servira aux filtres stricts) ---
        metadata = {
            "titre": titre,
            "auteur": auteur,
            "langue": row.get("Langue", "Non spécifié"),
            "lectorat": row.get("Lectorat", "Non spécifié"),
            "editeur": row.get("Editeur", "Non spécifié")
        }
        
        # Gestion des genres (fusion de Genre 1 et Genre 2)
        g1 = str(row.get("Genre 1") or "").strip()
        g2 = str(row.get("Genre 2") or "").strip()
        liste_genres = [g for g in [g1, g2] if g and g.lower() != "nan" and g != 'None']
        metadata["genre"] = ", ".join(liste_genres)
        
        # Gestion sécurisée des nombres (Pages, Année, Note)
        try:
            metadata["nb_pages"] = int(row.get("Nombre de pages", 0)) if row.get("Nombre de pages") else 0
            metadata["annee"] = int(row.get("Année", 0)) if row.get("Année") else 0
            
            note = str(row.get("Moyenne (/20)") or "0").replace(',', '.')
            metadata["note_moyenne"] = float(note)
        except ValueError:
            pass # Si la conversion échoue, on ignore ou on garde la valeur par défaut
            
        # Gestion de la date d'achat (Conversion en Timestamp pour les filtres mathématiques $gte / $lte)
        date_achat_brute = row.get("Date d'achat")
        if date_achat_brute and str(date_achat_brute).strip() != "None":
            try:
                # On ne garde que la date (DD/MM/YYYY) en ignorant l'heure
                date_str = str(date_achat_brute).split()[0]
                date_obj = dt.strptime(date_str, "%d/%m/%Y")
                # On stocke le timestamp (secondes) pour faciliter la comparaison ChromaDB
                metadata["date_achat_timestamp"] = int(date_obj.timestamp())
            except Exception as e:
                # Si la date est mal formatée, on passe
                pass
                
        # Création de l'objet Document LangChain
        doc = Document(page_content=page_content, metadata=metadata)
        documents.append(doc)
        
    print(f"✅ {len(documents)} documents préparés avec succès !")
    return documents

# Test de la fonction
# documents_bibliotheque = charger_et_preparer_csv("PAL_final.csv")
# print(documents_bibliotheque[0])

In [31]:
documents_bibliotheque = charger_et_preparer_csv("PAL_final.csv")
print(documents_bibliotheque[5])

Chargement des données depuis PAL_final.csv...
✅ 100 documents préparés avec succès !
page_content='Titre: 17 millimètres
Auteur: Médina Florence
Résumé: [Source: Livraddict] Un roman d’une justesse à couper le souffle sur l’avortement!
&quot;Comment peut-on se retrouver
En consultation pour une IVG
Avec une culotte
À licornes ?&quot;
Cet été, en camp d'ado, Mona et Liam ont fait l'amour pour la première fois. C'était bien.
Sauf que le préservatif a craqué.
Et qu'ils n'ont pas osé demander aux monos de les accompagner à la pharmacie pour demander une pilule du lendemain.
Mais après tout, Mona avait eu ses règles quelques jours plus tôt, alors... ça ne risquait rien, si ?
Six semaines plus tard, la jeune femme doit se rendre à l'évidence : elle est enceinte. Et d'après les informations qu'elle a glanées sur le net, elle héberge déjà un embryon de 17 millimètres.
C'est peu et c'est beaucoup, 17 millimètres. C'est trop quand on ne veut pas d'un enfant, là, tout de suite, à 16 ans.
Mona a 

# Embedding

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings

def creer_vectorstore_faiss(documents):
    print("⏳ Création des embeddings et de la base FAISS en cours...")
    
    # 1. Initialisation du modèle d'embeddings
    # "all-MiniLM-L6-v2" est léger, rapide et performant pour le français/anglais basique
    # embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    # embeddings = OllamaEmbeddings(model="nomic-embed-text")
    embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
    
    # 2. Création de la base FAISS à partir des documents préparés à l'Étape 1
    # vectorstore = FAISS.from_documents(documents, embeddings, distance_strategy=DistanceStrategy.COSINE)
    vectorstore = FAISS.from_documents(documents, embeddings)
    
    
    print(f"✅ Base FAISS créée avec succès ! ({vectorstore.index.ntotal} documents vectorisés)")
    
    # 3. (Optionnel) Sauvegarde locale pour éviter de recalculer les embeddings à chaque lancement
    chemin_sauvegarde = "faiss_index_bibliotheque"
    vectorstore.save_local(chemin_sauvegarde)
    print(f"💾 Base sauvegardée localement dans le dossier '{chemin_sauvegarde}'")
    
    return vectorstore

# Pour recharger la base plus tard sans tout recalculer :
# vectorstore = FAISS.load_local("faiss_index_bibliotheque", embeddings, allow_dangerous_deserialization=True)

In [66]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_ollama import OllamaEmbeddings

print("⏳ Chargement du modèle Ollama nomic-embed-text...")
# embeddings = OllamaEmbeddings(model="nomic-embed-text")
documents_bibliotheque = charger_et_preparer_csv("PAL_final.csv")
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

print("⏳ Création de la nouvelle base FAISS (Cosinus)...")
vectorstore = FAISS.from_documents(
    documents_bibliotheque, 
    embeddings,
    #distance_strategy=DistanceStrategy.COSINE
)

vectorstore.save_local("faiss_index_bibliotheque")
print("✅ Nouvelle base FAISS sauvegardée avec succès !")

⏳ Chargement du modèle Ollama nomic-embed-text...
Chargement des données depuis PAL_final.csv...
✅ 100 documents préparés avec succès !
⏳ Création de la nouvelle base FAISS (Cosinus)...
✅ Nouvelle base FAISS sauvegardée avec succès !


# Création de l'outil de recherche

In [34]:
from langchain.tools import tool
from typing import Optional
from datetime import datetime as dt

@tool
def chercher_livres_filtres(
    query: str, 
    auteur: Optional[str] = None, 
    langue: Optional[str] = None, 
    nb_pages_min: Optional[int] = None, 
    nb_pages_max: Optional[int] = None,
    note_min: Optional[float] = None,
    annee_min: Optional[int] = None,
    annee_max: Optional[int] = None,
    date_achat_apres: Optional[str] = None,
    date_achat_avant: Optional[str] = None,
    genre: Optional[str] = None,
    editeur: Optional[str] = None,
    lectorat: Optional[str] = None
) -> str:
    """
    Outil de recherche avancé pour interroger la bibliothèque de livres.
    
    Arguments:
    - query: Le sujet, thème ou mots-clés de la recherche (ex: "vampires", "romance à Londres").
    - auteur: Nom de l'auteur (ex: "J.K. Rowling").
    - langue: La langue du livre (ex: "fr", "en").
    - nb_pages_min / nb_pages_max: Filtre sur le nombre de pages (ex: min 300).
    - note_min: Note moyenne minimale sur 20 (ex: 15).
    - annee_min / annee_max: Année de publication (ex: min 2000).
    - date_achat_apres / date_achat_avant: Date d'achat au format strict 'YYYY-MM-DD' (ex: "2023-01-01").
    - genre: Le genre littéraire (ex: "Fantasy", "Thriller").
    - editeur: La maison d'édition.
    - lectorat: Le public visé (ex: "Adulte", "Jeunesse").
    """
    
    # --- LA MAGIE FAISS : Une fonction de filtre Python pure ---
    def filtre_faiss(metadata: dict) -> bool:
        # 1. Filtres textuels (correspondance partielle ou exacte)
        if auteur and auteur.lower() not in str(metadata.get("auteur", "")).lower(): return False
        if editeur and editeur.lower() not in str(metadata.get("editeur", "")).lower(): return False
        if lectorat and lectorat.lower() != str(metadata.get("lectorat", "")).lower(): return False
        if genre and genre.lower() not in str(metadata.get("genre", "")).lower(): return False
        
        # 2. Gestion de la langue (mapping hérité de ton code)
        if langue:
            mapping_langues = {"en": "Anglaise", "fr": "Française", "de": "Allemande"}
            langue_csv = mapping_langues.get(langue.lower(), langue)
            if metadata.get("langue") != langue_csv: return False

        # 3. Filtres numériques
        if nb_pages_min and metadata.get("nb_pages", 0) < nb_pages_min: return False
        if nb_pages_max and metadata.get("nb_pages", 0) > nb_pages_max: return False
        if note_min and metadata.get("note_moyenne", 0) < note_min: return False
        if annee_min and metadata.get("annee", 0) < annee_min: return False
        if annee_max and metadata.get("annee", 0) > annee_max: return False
        
        # 4. Filtres de dates (conversion YYYY-MM-DD -> Timestamp)
        if date_achat_apres:
            ts_apres = int(dt.strptime(date_achat_apres, "%Y-%m-%d").timestamp())
            if metadata.get("date_achat_timestamp", 0) < ts_apres: return False
            
        if date_achat_avant:
            ts_avant = int(dt.strptime(date_achat_avant, "%Y-%m-%d").timestamp())
            if metadata.get("date_achat_timestamp", 0) > ts_avant: return False

        # Si le document passe tous les tests, on le garde !
        return True

    print(f"DEBUG - Recherche RAG hybride lancée pour la query : '{query}'")

    try:
        # On demande les résultats AVEC leur score de pertinence
        results_with_scores = vectorstore.similarity_search_with_score(query, k=5, filter=filtre_faiss)
    except Exception as e:
        return f"Erreur technique lors de la recherche FAISS : {str(e)}"

    if not results_with_scores:
        return "Aucun livre trouvé correspondant exactement à ces critères dans la bibliothèque."

    textes_formates = []
    # On boucle sur les documents ET leurs scores
    for doc, score in results_with_scores:
        t = doc.metadata.get("titre", "Inconnu")
        a = doc.metadata.get("auteur", "Inconnu")
        
        # Affichage dans la console pour t'aider à débugger
        print(f"-> 🎯 Trouvé : {t} | Distance: {score:.2f}")
        
        # Optionnel : On peut forcer un seuil. Si la distance est trop grande (> 1.5 par exemple), on l'ignore.
        if score > 1.5: 
            continue 

        textes_formates.append(f"Titre: {t}\nAuteur: {a}\nRésumé: {doc.page_content}")
        
    if not textes_formates:
        return "Les livres trouvés n'étaient pas assez pertinents par rapport au sujet demandé."

    return "\n\n---\n\n".join(textes_formates)

In [56]:
from langchain.tools import tool
from typing import Optional
from datetime import datetime as dt

@tool
def chercher_livres_filtres(
    query: str, 
    auteur: Optional[str] = None, 
    langue: Optional[str] = None, 
    nb_pages_min: Optional[int] = None, 
    nb_pages_max: Optional[int] = None,
    note_min: Optional[float] = None,
    annee_min: Optional[int] = None,
    annee_max: Optional[int] = None,
    date_achat_apres: Optional[str] = None,
    date_achat_avant: Optional[str] = None,
    genre: Optional[str] = None,
    editeur: Optional[str] = None,
    lectorat: Optional[str] = None
) -> str:
    """Outil de recherche avancé pour interroger la bibliothèque de livres."""
    
    def filtre_faiss(metadata: dict) -> bool:
        if auteur and auteur.lower() not in str(metadata.get("auteur", "")).lower(): return False
        if editeur and editeur.lower() not in str(metadata.get("editeur", "")).lower(): return False
        if lectorat and lectorat.lower() != str(metadata.get("lectorat", "")).lower(): return False
        if genre and genre.lower() not in str(metadata.get("genre", "")).lower(): return False
        
        # On garde ton mapping qui marche très bien !
        if langue:
            mapping_langues = {"en": "Anglaise", "fr": "Française", "de": "Allemande"}
            langue_csv = mapping_langues.get(langue.lower(), langue)
            if metadata.get("langue") != langue_csv: return False
            
        if nb_pages_min and metadata.get("nb_pages", 0) < nb_pages_min: return False
        if nb_pages_max and metadata.get("nb_pages", 0) > nb_pages_max: return False
        if note_min and metadata.get("note_moyenne", 0) < note_min: return False
        if annee_min and metadata.get("annee", 0) < annee_min: return False
        if annee_max and metadata.get("annee", 0) > annee_max: return False
        if date_achat_apres:
            ts_apres = int(dt.strptime(date_achat_apres, "%Y-%m-%d").timestamp())
            if metadata.get("date_achat_timestamp", 0) < ts_apres: return False
        if date_achat_avant:
            ts_avant = int(dt.strptime(date_achat_avant, "%Y-%m-%d").timestamp())
            if metadata.get("date_achat_timestamp", 0) > ts_avant: return False
        return True

    try:
        # L'ASTUCE EST ICI : fetch_k=1000 force FAISS à ramener un maximum de documents avant de filtrer
        results_with_scores = vectorstore.similarity_search_with_score(
            query, 
            k=5, 
            filter=filtre_faiss,
            fetch_k=1000 
        )
    except Exception as e:
        return f"Erreur technique FAISS : {str(e)}"

    textes_formates = []
    
    for doc, distance in results_with_scores:
        # On ignore la limite de distance si la requête est juste le mot générique "livre"
        if query.lower().strip() != "livre" and distance > 15.0:
            continue
            
        t = doc.metadata.get("titre", "Inconnu")
        a = doc.metadata.get("auteur", "Inconnu")
        textes_formates.append(f"Titre: {t}\nAuteur: {a}\nRésumé: {doc.page_content}")
        
    if not textes_formates:
        return "Aucun livre pertinent n'a été trouvé pour ce sujet exact dans la bibliothèque."

    return "\n\n---\n\n".join(textes_formates)

In [57]:
# Test 1 : RAG Sémantique pur (sans filtres)
print(chercher_livres_filtres.invoke({"query": "meurtre"}))

# Test 2 : RAG Sémantique + Filtre métadonnée
# print(chercher_livres_filtres.invoke({"query": "un meurtre", "nb_pages_max": 300, "langue": "fr"}))

Titre: Autopsie, tome 1 : Whitechapel
Auteur: Maniscalco Kerri
Résumé: Titre: Autopsie, tome 1 : Whitechapel
Auteur: Maniscalco Kerri
Résumé: [Source: Livraddict] 1888, quartier Est de Londres. Depuis quelque temps, des meurtres sanglants et horribles touchent les femmes de petite vertu de Whitechapel. Une jeune femme, de bonne famille, en avance sur son temps, enquête au côté de son oncle, médecin légiste.

---

Titre: Autopsie, tome 2
Auteur: Maniscalco Kerri
Résumé: Titre: Autopsie, tome 2
Auteur: Maniscalco Kerri
Résumé: [Source: Livraddict] Bizarre murders are discovered in the castle of Prince Vlad the Impaler, otherwise known as Dracula. Could it be a copycat killer... or has the depraved prince been brought back to life?  Following the grief and horror of her discovery of Jack the Ripper's true identity, Audrey Rose Wadsworth has no choice but to flee London and its memories. Together with the arrogant yet charming Thomas Cresswell, she journeys to the dark heart of Romania, ho

# Étape 4 : L'adaptation des Schémas et de l'État pour LangGraph.

In [58]:
from typing import TypedDict, List, Optional
from pydantic import BaseModel, Field

# ---------------------------------------------------------
# 1. LE SCHÉMA DE DONNÉES (Pour forcer l'IA à extraire les bons filtres)
# ---------------------------------------------------------
class PlanRechercheLivre(BaseModel):
    # Le champ sémantique principal (pour le RAG FAISS)
    query: str = Field(
        ..., 
        description="Le thème principal, résumé ou mots-clés de l'histoire (ex: 'un meurtre dans un train', 'une romance', 'de la magie'). Si aucun thème n'est donné, mets 'livre'."
    )
    
    # Les filtres textuels (correspondance exacte ou partielle)
    auteur: Optional[str] = Field(None, description="Nom de l'auteur (ex: 'Eoin Colfer').")
    genre: Optional[str] = Field(None, description="Genre littéraire (ex: 'Fantasy', 'Polar').")
    editeur: Optional[str] = Field(None, description="Maison d'édition.")
    langue: Optional[str] = Field(None, description="Code langue ('fr' pour français, 'en' pour anglais).")
    lectorat: Optional[str] = Field(None, description="Public visé (ex: 'Adulte', 'Jeunesse', 'Young Adult').")
    
    # Les filtres numériques
    nb_pages_min: Optional[int] = Field(None, description="Nombre minimum de pages.")
    nb_pages_max: Optional[int] = Field(None, description="Nombre maximum de pages.")
    note_min: Optional[float] = Field(None, description="Note minimale sur 20 (ex: 15).")
    annee_min: Optional[int] = Field(None, description="Année de publication minimale (ex: 2010).")
    annee_max: Optional[int] = Field(None, description="Année de publication maximale.")
    
    # Les filtres temporels d'achat
    date_achat_apres: Optional[str] = Field(None, description="Acheté après cette date (Format absolu YYYY-MM-DD).")
    date_achat_avant: Optional[str] = Field(None, description="Acheté avant cette date (Format absolu YYYY-MM-DD).")

# ---------------------------------------------------------
# 2. L'ÉTAT DU GRAPHE (La mémoire partagée de l'Agent)
# ---------------------------------------------------------
class AgentState(TypedDict):
    question: str                          # La question initiale de l'utilisateur
    
    plan_recherche: Optional[dict]         # Le plan généré par l'LLM (format dict du PlanRechercheLivre)
    validation_errors: Optional[List[str]] # Liste des erreurs si le plan est illogique
    
    human_feedback: Optional[dict]         # Retour de l'utilisateur {'approved': True/False, 'new_question': str}
    
    resultats_livres: Optional[str]        # Les textes bruts retournés par notre outil FAISS
    reponse_finale: str                    # La réponse rédigée et amicale générée à la fin

# Noeuds du graphe 

In [59]:
import os
from langchain_mistralai import ChatMistralAI
from langchain_core.output_parsers import PydanticOutputParser

# 1. Configuration de la clé API
# Il est recommandé de la mettre en variable d'environnement, ou tu peux la passer directement.
os.environ["MISTRAL_API_KEY"] = "O4sxEU9DYRsTmG8h4DpIIZ7jyIafz91v" # Remplace par ta vraie clé

# 2. Initialisation du modèle Mistral
# "mistral-small-latest" est très rapide, pas cher et excellent pour ce genre de tâche de parsing. 
# Si tu veux le summum de l'intelligence, tu peux mettre "mistral-large-latest".
model = ChatMistralAI(
    model="mistral-small-latest", 
    temperature=0
)

# ... Le reste de tes fonctions (planning_node, validation_node, etc.) reste EXACTEMENT identique !

In [60]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# Initialise ton modèle (utilise celui qui marche le mieux chez toi, avec une température à 0 pour la rigueur)
from langchain_ollama import ChatOllama
# model = ChatOllama(model="llama3.2:3b", temperature=0, format="json") # ou "gemma3:4b", "llama3", etc.

# 1. NOEUD DE PLANIFICATION (Extraction des filtres via Pydantic)
def planning_node(state: AgentState):
    print("\n--- 🧠 PLANIFICATION DE LA RECHERCHE ---")
    question = state["question"]
    
    parser = PydanticOutputParser(pydantic_object=PlanRechercheLivre)
    
    prompt = f"""
    Tu es un algorithme de filtrage de bibliothèque de haut niveau.
    Ta tâche est de séparer le SUJET de la recherche des FILTRES techniques.

    Question de l'utilisateur : "{question}"
    
    RÈGLES STRICTES :
    1. Si l'utilisateur parle d'une histoire, d'un thème ou d'un résumé, mets-le dans 'query'. S'il n'y a pas de thème précis, mets 'livre'.
    2. Les genres (romance, fantasy, polar...) vont dans 'genre'.
    3. Les langues explicites (français, anglais) vont dans 'langue' (fr, en).
    4. Pour le nombre de pages ("livre court" -> max 300, "gros livre" -> min 500).
    5. N'invente AUCUNE information. Si ce n'est pas dans la phrase, mets null.
    
    Réponds UNIQUEMENT par le JSON généré.
    {parser.get_format_instructions()}
    """
    
    try:
        # On force le modèle à sortir un JSON valide
        reponse = model.invoke(prompt)
        # Parfois les LLM rajoutent "```json" autour, le parseur gère souvent ça, mais on peut sécuriser
        plan = parser.parse(reponse.content)
        return {"plan_recherche": plan.model_dump(), "validation_errors": None}
    except Exception as e:
        print(f"⚠️ Erreur de parsing LLM : {e}")
        # Fallback basique si le modèle a échoué à faire du JSON
        return {"plan_recherche": {"query": question}, "validation_errors": ["Erreur de parsing JSON par l'IA."]}

# 2. NOEUD DE VALIDATION (Vérification logique)
def validation_node(state: AgentState):
    print("--- 🛡️ VALIDATION DES DONNÉES ---")
    plan = state.get("plan_recherche")
    errors = state.get("validation_errors") or []
    
    if plan:
        # Vérification des pages
        p_min = plan.get("nb_pages_min")
        p_max = plan.get("nb_pages_max")
        if p_min and p_max and p_min > p_max:
            errors.append(f"Erreur : Pages Min ({p_min}) > Pages Max ({p_max}).")
            
        # Vérification des années
        a_min = plan.get("annee_min")
        a_max = plan.get("annee_max")
        if a_min and a_max and a_min > a_max:
            errors.append(f"Erreur : Année Min ({a_min}) > Année Max ({a_max}).")
            
    return {"validation_errors": errors if errors else None}

# 3. NOEUD D'APPROBATION HUMAINE
def human_approval_node(state: AgentState):
    # Si on a déjà un feedback, on passe
    if state.get("human_feedback"): return state
    
    plan = state.get("plan_recherche", {})
    print(f"\n--- 👮 VALIDATION DES CRITÈRES PAR L'UTILISATEUR ---")
    
    for k, v in plan.items():
        if v is not None:
            print(f"• {k}: {v}")
            
    decision = input("\nCes critères sont-ils corrects ? (oui/non) : ").strip().lower()
    
    if decision in ['oui', 'o', 'yes', 'y']:
        return {"human_feedback": {"approved": True}}
    else:
        new_q = input("Veuillez reformuler votre demande : ")
        return {"human_feedback": {"approved": False, "new_question": new_q}}

# 4. NOEUD D'EXÉCUTION (Appel de l'outil FAISS)
def execution_node(state: AgentState):
    print("\n--- 🚀 EXÉCUTION DE LA RECHERCHE (FAISS) ---")
    plan = state["plan_recherche"]
    
    try:
        # On passe directement le dictionnaire de filtres à notre outil !
        resultats = chercher_livres_filtres.invoke(plan)
    except Exception as e:
        resultats = f"Erreur technique lors de la requête : {e}"

    return {"resultats_livres": resultats}

# 5. NOEUD DE SYNTHÈSE (Rédaction finale)
def synthesis_node(state: AgentState):
    print("\n--- ✍️ RÉDACTION DE LA RÉPONSE ---")
    question = state["question"]
    contexte_livres = state.get("resultats_livres", "Aucun résultat.")
    
    prompt = f"""
    Tu es un bibliothécaire personnel très chaleureux et cultivé.
    
    DEMANDE INITIALE DE L'UTILISATEUR : "{question}"
    
    LIVRES TROUVÉS DANS SA BIBLIOTHÈQUE :
    --------------------
    {contexte_livres}
    --------------------
    
    CONSIGNES :
    1. Si le contexte dit "Aucun livre trouvé", dis-lui gentiment que tu n'as rien trouvé qui correspond exactement à ces critères dans sa PAL (Pile à Lire). Ne propose PAS de livres inventés.
    2. Si des livres sont trouvés, présente-les de manière enthousiaste en expliquant pourquoi ils correspondent à sa demande.
    3. Ne mentionne pas les scores ou les rouages techniques.
    
    Rédige ta réponse finale :
    """
    
    try:
        reponse = model.invoke(prompt)
        return {"reponse_finale": reponse.content}
    except Exception as e:
        return {"reponse_finale": "Désolé, j'ai rencontré un problème pour rédiger ma réponse."}

# Etape de test

In [61]:
from langgraph.graph import StateGraph, END

# ---------------------------------------------------------
# 1. INITIALISATION DU GRAPHE
# ---------------------------------------------------------
graph = StateGraph(AgentState)

# 2. AJOUT DES NOEUDS
graph.add_node("planifier", planning_node)
graph.add_node("valider", validation_node)
graph.add_node("humain", human_approval_node)
graph.add_node("chercher", execution_node)
graph.add_node("repondre", synthesis_node)

# ---------------------------------------------------------
# 3. DÉFINITION DU FLUX (EDGES ET ROUTEURS)
# ---------------------------------------------------------
# Le point d'entrée est toujours la planification
graph.set_entry_point("planifier")

# Planifier -> Valider
graph.add_edge("planifier", "valider")

# Routeur après Validation
def validation_router(state: AgentState):
    if state.get("validation_errors"):
        print("❌ Erreur de validation, arrêt de l'agent.")
        return END 
    return "humain" 

graph.add_conditional_edges("valider", validation_router)

# Routeur après Approbation Humaine
def human_router(state: AgentState):
    feedback = state.get("human_feedback", {})
    if feedback.get("approved"):
        return "chercher"
    elif feedback.get("new_question"):
        # On met à jour la question et on efface l'ancien plan/feedback pour recommencer
        state["question"] = feedback["new_question"]
        state["plan_recherche"] = None
        state["human_feedback"] = None
        return "planifier"
    return END

graph.add_conditional_edges("humain", human_router)

# Chercher -> Répondre -> FIN
graph.add_edge("chercher", "repondre")
graph.add_edge("repondre", END)

# ---------------------------------------------------------
# 4. COMPILATION
# ---------------------------------------------------------
app_biblio = graph.compile()
print("✅ Agent LangGraph compilé et prêt à l'emploi !")



✅ Agent LangGraph compilé et prêt à l'emploi !


In [53]:
# ---------------------------------------------------------
# 5. TEST COMPLET
# ---------------------------------------------------------
print("\n" + "="*50)
print("TEST DE L'AGENT DE BIBLIOTHÈQUE")
print("="*50)

# Remplace par la question de ton choix !
inputs = {"question": "Je cherche un gros livre de fantasy (plus de 400 pages)."}

for step in app_biblio.stream(inputs):
    # Ce petit bout de code permet juste d'afficher joliment quelle étape vient de se terminer
    etape_nom = list(step.keys())[0]
    if etape_nom == "repondre":
        print("\n🤖 RÉPONSE FINALE DE L'AGENT :")
        print(step["repondre"]["reponse_finale"])


TEST DE L'AGENT DE BIBLIOTHÈQUE

--- 🧠 PLANIFICATION DE LA RECHERCHE ---
--- 🛡️ VALIDATION DES DONNÉES ---

--- 👮 VALIDATION DES CRITÈRES PAR L'UTILISATEUR ---
• query: livre
• genre: fantasy
• nb_pages_min: 500

--- 🚀 EXÉCUTION DE LA RECHERCHE (FAISS) ---
✅ Gardé : Chroniques de l'Empire, intégrale, tome 1 : La Marche du Nord, Un port au Sud (Score: 5.60)

--- ✍️ RÉDACTION DE LA RÉPONSE ---

🤖 RÉPONSE FINALE DE L'AGENT :
Ah, quelle excellente demande ! Je suis ravi de vous proposer un livre qui correspond parfaitement à vos critères : **"Chroniques de l'Empire, intégrale, tome 1 : La Marche du Nord, Un port au Sud"** de Georges Foveau.

Avec plus de 400 pages, ce roman vous plongera dans un univers de fantasy riche et captivant. Vous y suivrez les aventures de Soze, un fonctionnaire impérial envoyé dans des régions sauvages et mystérieuses, où les lois de l'Empire ne s'appliquent plus. Entre intrigues politiques, forêts médiévales et complots obscurs, ce livre est une véritable pépit

In [63]:
inputs = {"question": "Je cherche une Romance en anglais de moins de 350 pages."}

for step in app_biblio.stream(inputs):
    # Ce petit bout de code permet juste d'afficher joliment quelle étape vient de se terminer
    etape_nom = list(step.keys())[0]
    if etape_nom == "repondre":
        print("\n🤖 RÉPONSE FINALE DE L'AGENT :")
        print(step["repondre"]["reponse_finale"])


--- 🧠 PLANIFICATION DE LA RECHERCHE ---
--- 🛡️ VALIDATION DES DONNÉES ---

--- 👮 VALIDATION DES CRITÈRES PAR L'UTILISATEUR ---
• query: livre
• genre: Romance
• langue: en
• nb_pages_max: 350

--- 🚀 EXÉCUTION DE LA RECHERCHE (FAISS) ---

--- ✍️ RÉDACTION DE LA RÉPONSE ---

🤖 RÉPONSE FINALE DE L'AGENT :
**Réponse :**

Oh, quelle merveilleuse demande ! J’ai trouvé un roman qui correspond parfaitement à ce que vous cherchez : **"Again Again" d’E. Lockhart** !

C’est une romance pleine de surprises, à la fois tendre et profonde, qui explore les thèmes de l’amour, du pardon et de la découverte de soi. Avec moins de 350 pages, c’est une lecture captivante qui vous emportera dans l’histoire d’Adelaide, une héroïne attachante qui traverse un été de possibilités infinies. Entre rebondissements et émotions, ce livre est une pépite romantique qui saura vous charmer.

Si vous aimez les histoires qui mêlent romance, introspection et une touche de magie, celui-ci est fait pour vous ! 📖✨

Dites-moi 

In [64]:
inputs = {"question": "Je cherche mes livres sur Artemis Fowl."}

for step in app_biblio.stream(inputs):
    # Ce petit bout de code permet juste d'afficher joliment quelle étape vient de se terminer
    etape_nom = list(step.keys())[0]
    if etape_nom == "repondre":
        print("\n🤖 RÉPONSE FINALE DE L'AGENT :")
        print(step["repondre"]["reponse_finale"])


--- 🧠 PLANIFICATION DE LA RECHERCHE ---
--- 🛡️ VALIDATION DES DONNÉES ---

--- 👮 VALIDATION DES CRITÈRES PAR L'UTILISATEUR ---
• query: Artemis Fowl

--- 🚀 EXÉCUTION DE LA RECHERCHE (FAISS) ---

--- ✍️ RÉDACTION DE LA RÉPONSE ---

🤖 RÉPONSE FINALE DE L'AGENT :
**Réponse chaleureuse :**

Ah, quelle excellente série vous avez là ! Je suis ravi de vous annoncer que votre bibliothèque regorge d’aventures palpitantes autour d’Artemis Fowl. Voici les deux tomes que j’ai dénichés pour vous :

1. **"Le Complexe d’Atlantis" (Tome 7)** – Un Artemis Fowl plus mature, mais pas moins ingénieux, se lance dans une mission pour sauver la planète… tout en luttant contre ses propres démons. Entre robots menaçants, pieuvres géantes et une personnalité qui change, ce tome est un mélange parfait d’action, d’humour et de suspense !

2. **"Le Paradoxe du temps" (Tome 6)** – Quand la mère d’Artemis tombe malade, notre jeune génie n’a d’autre choix que de voyager dans le temps pour trouver un remède. Mais att

In [65]:
inputs = {"question": "Je cherche un roman écrit par Cathrine Arnaud."}

for step in app_biblio.stream(inputs):
    # Ce petit bout de code permet juste d'afficher joliment quelle étape vient de se terminer
    etape_nom = list(step.keys())[0]
    if etape_nom == "repondre":
        print("\n🤖 RÉPONSE FINALE DE L'AGENT :")
        print(step["repondre"]["reponse_finale"])


--- 🧠 PLANIFICATION DE LA RECHERCHE ---
--- 🛡️ VALIDATION DES DONNÉES ---

--- 👮 VALIDATION DES CRITÈRES PAR L'UTILISATEUR ---
• query: livre
• auteur: Cathrine Arnaud

--- 🚀 EXÉCUTION DE LA RECHERCHE (FAISS) ---

--- ✍️ RÉDACTION DE LA RÉPONSE ---

🤖 RÉPONSE FINALE DE L'AGENT :
**Réponse du bibliothécaire :**

Ah, Cathrine Arnaud ! Quelle autrice captivante ! Je suis ravi de vous proposer trois titres de sa série *À la place du cœur*, qui correspondent parfaitement à votre demande. Ces romans mêlent avec une grande sensibilité l'histoire d'un amour naissant et les échos des attentats de 2015 en France, offrant une réflexion poignante sur la jeunesse, la résilience et l'amour face à l'horreur.

- **Tome 1** : *À la place du cœur* plonge dans les six jours intenses de Caumes, 17 ans, qui vit son premier amour avec Esther, tandis que la France est frappée par les attentats. Un récit vibrant entre passion adolescente et réalité brutale.

- **Tome 2** : *À la place du cœur* (tome 2) suit 